# Profiling with Nsight (Phase 10)

Runs `profiling/*.sh` on a Colab GPU and downloads the reports. Read `docs/10_nsight_profiling.md` first: it explains every number these reports contain.

Use the same GPU type as the benchmark run (T4) so profiles and benchmarks describe the same hardware.

In [ ]:
# 1. Get the code (same as colab_run.ipynb: git clone OR upload the zip), then build.
# from google.colab import files; uploaded = files.upload(); zip_name = next(iter(uploaded)); !unzip -q -o "{zip_name}" -d /content/
%cd /content/cuda-transformer-performance
!sed -i 's/\r$//' scripts/*.sh profiling/*.sh
!cmake -S . -B build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build build -j 2>&1 | tail -n 3
!nvidia-smi --query-gpu=name --format=csv,noheader

In [ ]:
# 2. Are the profilers installed? (They ship with the CUDA toolkit; Colab usually has them.)
!which ncu nsys || ls /usr/local/cuda/bin | grep -E 'ncu|nsys' || echo 'profiler(s) missing: see docs/10 section 2'
!ncu --version | tail -n 1; nsys --version

In [ ]:
# 3. Registers / shared memory / spills per kernel (no profiler needed)
!bash profiling/resource_usage.sh

In [ ]:
# 4. Nsight Systems timeline (all cases, 5 repetitions each)
!bash profiling/nsys_timeline.sh all

In [ ]:
# 5. Nsight Compute, one case at a time (each takes about 1-3 minutes)
for case in ["gemm", "precision", "softmax", "layernorm", "attention", "vector_add"]:
    print(f"===== {case} =====")
    !bash profiling/ncu_kernels.sh {case}

In [ ]:
# 6. Download every report (text, CSV and the binary reports for the Nsight GUIs)
!zip -q -r profiling_reports.zip profiling/reports
from google.colab import files
files.download("profiling_reports.zip")